# DDoS Attack Detection - Exploratory Data Analysis

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

try:
    plt.style.use('seaborn-v0_8-darkgrid')
except OSError:
    plt.style.use('seaborn-darkgrid')

sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
print('Libraries imported successfully!')

In [ ]:
np.random.seed(42)
n_samples = 10000

normal_data = {
    'packets_per_second': np.random.normal(5000, 1000, n_samples // 2),
    'bytes_per_second':   np.random.normal(50e6,  10e6, n_samples // 2),
    'flows_per_second':   np.random.normal(1000,  200,  n_samples // 2),
    'unique_src_ips':     np.random.normal(100,   20,   n_samples // 2),
    'entropy_src_ip':     np.random.normal(3.5,   0.5,  n_samples // 2),
    'entropy_dst_ip':     np.random.normal(2.0,   0.3,  n_samples // 2),
    'tcp_ratio':          np.random.beta(10, 5,         n_samples // 2),
    'syn_ratio':          np.random.beta(2,  8,         n_samples // 2),
    'is_attack': 0,
}
attack_data = {
    'packets_per_second': np.random.exponential(50000, n_samples // 2),
    'bytes_per_second':   np.random.exponential(500e6, n_samples // 2),
    'flows_per_second':   np.random.exponential(5000,  n_samples // 2),
    'unique_src_ips':     np.random.exponential(500,   n_samples // 2),
    'entropy_src_ip':     np.random.normal(1.5,  0.5,  n_samples // 2),
    'entropy_dst_ip':     np.random.normal(1.0,  0.3,  n_samples // 2),
    'tcp_ratio':          np.random.beta(5, 5,         n_samples // 2),
    'syn_ratio':          np.random.beta(8, 2,         n_samples // 2),
    'is_attack': 1,
}
df = pd.concat([pd.DataFrame(normal_data), pd.DataFrame(attack_data)], ignore_index=True)
print(f'Dataset shape: {df.shape}')
df.head()

In [ ]:
features_to_plot = ['packets_per_second', 'bytes_per_second', 'flows_per_second',
                    'unique_src_ips', 'entropy_src_ip', 'tcp_ratio']

# --- Distribution plots ---
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()
for i, feature in enumerate(features_to_plot):
    for at, color, label in [(0,'green','Normal'),(1,'red','Attack')]:
        axes[i].hist(df[df['is_attack']==at][feature], bins=50,
                     alpha=0.5, color=color, label=label, density=True)
    axes[i].set_title(f'Distribution of {feature}')
    axes[i].set_yscale('log')
    axes[i].legend()
plt.tight_layout()
plt.show()

In [ ]:
feature_cols = [c for c in df.columns if c != 'is_attack']
correlation_matrix = df[feature_cols].corr(numeric_only=True)

plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

label_corr = df[feature_cols].corrwith(df['is_attack']).sort_values(ascending=False)
print('\nFeatures most correlated with attack label:')
print(label_corr)

In [ ]:
def detect_outliers_iqr(data, column, multiplier=1.5):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lo = Q1 - multiplier * IQR
    hi = Q3 + multiplier * IQR
    return len(data[(data[column] < lo) | (data[column] > hi)]), lo, hi

df_normal = df[df['is_attack'] == 0]
df_attack = df[df['is_attack'] == 1]

print('Outlier Detection (IQR from normal traffic only):')
print('=' * 70)
for feature in features_to_plot:
    n_out, lo, hi = detect_outliers_iqr(df_normal, feature)
    n_atk = int(((df_attack[feature] < lo) | (df_attack[feature] > hi)).sum())
    pct   = n_atk / len(df_attack) * 100
    print(f'{feature:22s}: {n_out:5d} normal outliers | '
          f'{n_atk:5d}/{len(df_attack)} attacks flagged ({pct:.1f}%)')
    print(f'  Normal IQR range: [{lo:.2f}, {hi:.2f}]')
    print()

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

X = df.drop('is_attack', axis=1)
y = df['is_attack']

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X, y)

fi = pd.DataFrame({'feature': X.columns,
                   'importance': rf.feature_importances_}).sort_values('importance', ascending=False)
plt.figure(figsize=(10, 6))
plt.barh(fi['feature'], fi['importance'], color='steelblue')
plt.xlabel('Importance')
plt.title('Feature Importance for DDoS Detection')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()
print('Top 5:')
print(fi.head())